In [19]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

# Define the state structure
class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]

# Define node a
def node_a(state: State):
    return {"foo": "a", "bar": ["a"]}

# Define node b
def node_b(state: State):
    return {"foo": "b", "bar": ["b"]}

# Initialize the StateGraph with the State schema
workflow = StateGraph(State)

# Add nodes to the graph
workflow.add_node("node_a", node_a)
workflow.add_node("node_b", node_b)

# Define edges and flow sequence
workflow.add_edge(START, "node_a")
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", END)

# Set up the memory checkpointer and compile the graph
checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

In [20]:
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

# Pass the config into invoke here:
graph.invoke({"foo": "", "bar": []}, config)

graph.get_state(config)

StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b3952-4808-6b42-8002-1e6ff91605b8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-18T19:14:16.909874+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b3952-4802-6eee-8001-7c5bc7506335'}}, tasks=(), interrupts=())

In [23]:
config = {"configurable": {"thread_id": "1", "checkpoint_id":"1f1b3952-4802-6eee-8001-7c5bc7506335"}}
graph.get_state(config)

StateSnapshot(values={'foo': 'a', 'bar': ['a']}, next=('node_b',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1b3952-4802-6eee-8001-7c5bc7506335'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-09-18T19:14:16.907509+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b3952-47fc-6371-8000-ac186033cfae'}}, tasks=(PregelTask(id='4af41c7f-4d6a-cc70-a0a8-3a03a11488aa', name='node_b', path=('__pregel_pull', 'node_b'), error=None, interrupts=(), state=None, result={'foo': 'b', 'bar': ['b']}),), interrupts=())